# Topic 3: Basic Document Classification (Part 1)

## Overview
In labs this week (and next), the focus will be on the application of sentiment analysis. You will be using a corpus of **movie reviews**.

You will be exploring various techniques that can be used to classify the sentiment of the movie reviews as either positive or negative.

You will be developing your own **Word List** and **Naïve Bayes** classifiers and then comparing them to the **NLTK Naïve Bayes** classifier.

First, we will need to download the movie_review corpus.

In [2]:
import nltk
nltk.download('movie_reviews')
nltk.download('stopwords')

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

The movie_reviews corpus reader provides a number of useful methods:
   * .categories()
   * .fileids()
   * .words()
   
First, we can use `.categories()` to check the set of labels with which the reviews have been labelled

In [3]:
from nltk.corpus import movie_reviews

print(movie_reviews.categories())

['neg', 'pos']


We can use `.fileids()` to get all of the file names associated with a particular category.

In [4]:
pos_review_ids=movie_reviews.fileids('pos')
neg_review_ids=movie_reviews.fileids('neg')

print("The number of positive reviews is {}".format(len(pos_review_ids)))
print("The number of negative reviews is {}".format(len(neg_review_ids)))

The number of positive reviews is 1000
The number of negative reviews is 1000


In [5]:
print(pos_review_ids[1])

pos/cv001_18431.txt


We can use `.words()` to get back word-tokenised reviews.  The argument to `.words()` is the file id of an individual review.

In [6]:
print(movie_reviews.words(pos_review_ids[2]))

['you', "'", 've', 'got', 'mail', 'works', 'alot', ...]


In [7]:
#find out what type it is
type(movie_reviews.words(pos_review_ids[0]))

nltk.corpus.reader.util.StreamBackedCorpusView

Note, the object returned by `movie_reviews.words()` looks a lot like a list (and behaves a lot like a list) - but it is actually a `StreamBackedCorpusView`.  This essentially means it is not necessarily all in memory  - it is retrieved from disk as needed.  If you want to see all of the words at once then you can convert it to a list using the `list()` constructor.  

In [8]:
#it behaves a like a list anyway so we dont need to worry about it too much

print(list(movie_reviews.words(pos_review_ids[1])))

['every', 'now', 'and', 'then', 'a', 'movie', 'comes', 'along', 'from', 'a', 'suspect', 'studio', ',', 'with', 'every', 'indication', 'that', 'it', 'will', 'be', 'a', 'stinker', ',', 'and', 'to', 'everybody', "'", 's', 'surprise', '(', 'perhaps', 'even', 'the', 'studio', ')', 'the', 'film', 'becomes', 'a', 'critical', 'darling', '.', 'mtv', 'films', "'", '_election', ',', 'a', 'high', 'school', 'comedy', 'starring', 'matthew', 'broderick', 'and', 'reese', 'witherspoon', ',', 'is', 'a', 'current', 'example', '.', 'did', 'anybody', 'know', 'this', 'film', 'existed', 'a', 'week', 'before', 'it', 'opened', '?', 'the', 'plot', 'is', 'deceptively', 'simple', '.', 'george', 'washington', 'carver', 'high', 'school', 'is', 'having', 'student', 'elections', '.', 'tracy', 'flick', '(', 'reese', 'witherspoon', ')', 'is', 'an', 'over', '-', 'achiever', 'with', 'her', 'hand', 'raised', 'at', 'nearly', 'every', 'question', ',', 'way', ',', 'way', ',', 'high', '.', 'mr', '.', '"', 'm', '"', '(', 'matt

## Creating training and testing sets
You will be training and testing various document classifiers. It is essential that the data used in the testing phase is not used during the training phase, since this can lead to overestimating performance.

We now introduce the `split_data` function (defined in the cell below) which can be used to get separate **training** and **testing** sets.

> Look through the code in the following cell, reading the comments and making sure that you understand each line.

In [9]:
import random # have a look at the documentation at https://docs.python.org/3/library/random.html

#random.seed is an important function

# ratio of 70% uses a python library called python

def split_data(data, ratio=0.7): # when the second argument is not given, it defaults to 0.7
    """
    Given collection of items and ratio:
     - partitions the collection into training and testing, where the proportion in training is ratio,

    :param data: A list (or generator) of documents or doc ids
    :param ratio: The proportion of training documents (default 0.7)
    :return: a pair (tuple) of lists where the first element of the
            pair is a list of the training data and the second is a list of the test data.
    """

    n = len(data)  #Found out number of samples present.  data could be a list or a generator
    train_indices = random.sample(range(n), int(n * ratio))          #Randomly select training indices
    test_indices = list(set(range(n)) - set(train_indices))   #Other items are testing indices

    train = [data[i] for i in train_indices]           #Use training indices to select data
    test = [data[i] for i in test_indices]             #Use testing indices to select data

    return (train, test)                       #Return split data


Now we can use this function to create training and testing data.  First, we need to create 4 lists:
    * file ids  of positive docs to go in the training data
    * file ids of positive docs to go in the testing data
    * file ids of negative docs to go in the training data
    * file ids of negative docs to go in the testing data

In [10]:
random.seed(42)  #set the random seeds so these random splits are always the same
pos_train_ids, pos_test_ids = split_data(pos_review_ids)
neg_train_ids, neg_test_ids = split_data(neg_review_ids)


In [11]:
print(pos_train_ids[0])

pos/cv654_18246.txt


Now, we want to create our labelled data sets.   We need to associate each review with its label so that later we can shuffle up all of the training data (and the testing data)

### Exercise 1
Write some python code which will construct a training set (`training`) and a test set (`testing`) from the data.  Each set should be a list of pairs where each pair (tuple) is a list of words and a label, as below:

<code>[([list,of,words],'label'),([list,of,words],'label'),...]</code>

Hint:  You can do this with 4 list comprehensions and list concatenation.

Check the size of `training` and `testing`.  Using a 70\% split, how many should be in each?

In [12]:
#test and experiment
movie_reviews.words(pos_train_ids[0])
print ('the words are', (list((movie_reviews.words(pos_train_ids[0]))),'pos'))

the words are (['the', 'summer', 'movie', 'season', 'is', 'always', 'the', 'biggest', 'and', 'brightest', 'season', 'for', 'movies', 'throughout', 'the', 'year', '.', 'we', 'are', 'introduced', 'to', 'blockbuster', 'hits', 'that', 'are', 'usually', 'trashed', 'by', 'critics', ',', 'loved', 'by', 'the', 'people', ',', 'and', 'make', 'tons', 'of', 'money', 'at', 'the', 'box', 'office', '.', 'this', 'year', ',', 'the', 'tradition', 'continues', 'with', 'an', 'update', 'on', 'the', 'universal', '1932', 'classic', ',', 'the', 'mummy', '.', 'i', 'have', 'always', 'been', 'a', 'sucker', 'for', 'this', 'kind', 'of', 'monster', 'movies', ',', 'and', 'i', 'always', 'seem', 'to', 'be', 'the', 'only', 'critic', 'who', 'really', 'likes', 'them', '.', 'that', 'tradition', 'continues', 'as', 'well', 'with', 'this', 'fun', 'special', 'effects', 'extravaganza', '.', 'in', 'the', 'mummy', ',', 'the', 'up', 'and', 'coming', 'brendan', 'fraser', 'stars', 'as', 'the', 'adventurous', 'explorer', 'rick', 'o'

In [13]:

#Positive Training Set
pos_train =[(list((movie_reviews.words(doc_id))),'pos')
  for doc_id in pos_train_ids
]
print('the positive training set total is',len(pos_train))

#Negative Training Set
neg_train =[(list((movie_reviews.words(doc_id))),'neg')
  for doc_id in neg_train_ids
]
print('the negative training set total is',len(neg_train))

#Postive Testing Set
pos_test =[(list((movie_reviews.words(doc_id))),'pos')
  for doc_id in pos_test_ids
]
print('the positive test set total is',len(pos_test))


#Negative Testing Set
neg_test =[(list((movie_reviews.words(doc_id))),'neg')
  for doc_id in neg_test_ids
]
print('the negative test set total is',len(neg_test))



the positive training set total is 700
the negative training set total is 700
the positive test set total is 300
the negative test set total is 300


In [14]:
training = pos_train + neg_train
testing = pos_test + neg_test

print('the length of total training set is',len(training))
print('the length of the total testing set is',len(testing))

the length of total training set is 1400
the length of the total testing set is 600


## Document Representations

Currently, each review / document is represented as a list of tokens.  In many simple applications, the order of words in a document is deemed irrelevant and we use a bag-of-words representation of the document.  We can create a bag-of-words using a dictionary (as we did in Lab_2_2 when considering the size of the vocabulary) or we can use a library function such as FreqDist from nltk.probability (or Counter from Collections).  In the cell below, I generate the bag-of-words for the first review in the training set using nltk's FreqDist.  You can think of this as like a dictionary but with extra benefits.  For example, later on in the lab, we will see it has useful methods which allow the document representations to be added and subtracted.

In [15]:
from nltk.probability import FreqDist

doc1 = FreqDist(training[0][0])
doc1

FreqDist({'the': 75, ',': 56, '.': 40, 'of': 25, 'and': 22, 'to': 18, 'in': 16, 'is': 15, 'a': 13, 'that': 12, ...})

### Exercise 2.1

Write code to use FreqDist to construct a bag-of-words representation for each document in the training and testing sets.  Store the results in two lists, `training_basic` and `testing_basic`.  Don't lose the annotations as to whether each review is positive or negative!  

In [16]:
training_basic = [(FreqDist(words), label)
  for (words, label) in training
]


testing_basic =[(FreqDist (words), label)
    for (words,label) in testing

]


print(training_basic[10])
print(testing_basic[10])


(FreqDist({'the': 97, ',': 80, '.': 76, '"': 52, 'of': 46, 'to': 44, 'a': 39, 'in': 37, "'": 33, 'and': 31, ...}), 'pos')
(FreqDist({',': 35, 'of': 21, '.': 21, 'the': 18, 'a': 17, 'and': 16, 'flanagan': 10, '"': 10, 'his': 9, 'by': 6, ...}), 'pos')


You will notice of course that many of the words in your representations of documents are punctuation and stopwords.  This is because we haven't done any pre-processing of the wordlists.

### Exercise 2.2

Decide which of the following pre-processing steps to apply to the word lists:-
* case normalisation
* number normalisation
* punctuation removal
* stopword removal
* stemmming / lemmatisation


Apply these preprocessing steps to the original wordlist representations (stored in `training` and `testing`).  Then recreate the bag-of-words representations, storing the results in `training_norm` and `testing_norm`

In [17]:
from nltk.corpus import stopwords
from nltk.probability import FreqDist

stop = stopwords.words("english")

training_norm = []
for review, label in training:
    normed = []
    for token in review:
        # Lowercase and filter out stopwords/punctuation
        token_lower = token.lower()
        if len(token_lower) > 1 and token_lower not in stop and token_lower.isalnum():
            normed.append(token_lower)
    training_norm.append((FreqDist(normed), label))

print(f"Total training_norm size: {len(training_norm)}")
print("Sample training item:", training_norm[0])

testing_norm = []
for review, label in testing:
    normed = []
    for token in review:
        token_lower = token.lower()
        if len(token_lower) > 1 and token_lower not in stop and token_lower.isalnum():
            normed.append(token_lower)
    testing_norm.append((FreqDist(normed), label))

print(f"Total testing_norm size: {len(testing_norm)}")

Total training_norm size: 1400
Sample training item: (FreqDist({'mummy': 12, 'film': 6, 'evelyn': 5, 'egypt': 5, 'like': 5, 'ancient': 5, 'one': 5, 'always': 4, 'special': 4, 'effects': 4, ...}), 'pos')
Total testing_norm size: 600


I lowercased the words so that variants such as `Great` and `great` contribute to the same count. This loses some emphasis from capitalisation. I also removed punctuation-only tokens, such as `!`, although these can carry sentiment or emphasis too.

I kept numbers and stopwords because they can provide useful context or change sentiment, particularly words such as `not`. I did not use stemming or lemmatisation because I wanted to preserve the original word forms.

These are sensible starting choices, but their effect on classification performance would need testing.

## Creating word lists
The next section will explain how to use a sentiment classifier that bases its decisions on word lists. The classifier requires a list of words indicating positive sentiment, and a second list of words indicating negative sentiment. Given positive and negative word lists, a document's overall sentiment is determined based on counts of occurrences of words that occur in the two lists. In this section we are concerned with the creation of the word lists. We will be considering both hand-crafted lists and automatically generated lists.

### Exercise 3.1

- Create a reasonably long hand-crafted list of words that you think indicate positive sentiment.
- Create a reasonably long hand-crafted list of words that indicate negative sentiment.

Use the following cells to store these lists in the variables `my_positive_word_list` and `my_negative_word_list`.

In [18]:
my_positive_word_list = [
    "good", "great", "lovely", "excellent", "amazing", "wonderful", "fantastic",
    "brilliant", "superb", "outstanding", "masterpiece", "awesome", "beautiful",
    "perfect", "enjoyable", "funny", "love", "hilarious", "engaging", "smart"
]

my_negative_word_list = [
    "bad", "terrible", "awful", "boring", "worst", "waste", "dreadful",
    "horrible", "stupid", "dull", "disappointing", "pointless", "annoying",
    "mess", "poor", "hate", "badly", "lame", "predictable", "cliche"
]

print(f"Positive words ({len(my_positive_word_list)}): {my_positive_word_list[:5]}...")
print(f"Negative words ({len(my_negative_word_list)}): {my_negative_word_list[:5]}...")

Positive words (20): ['good', 'great', 'lovely', 'excellent', 'amazing']...
Negative words (20): ['bad', 'terrible', 'awful', 'boring', 'worst']...


Now lets see how often each of those words occurs in total in our positive and negative training data.  First, lets create a total of the FreqDists for positive data and for negative data.  As these are FreqDists (rather than simple dictionaries), we can do this as follows:

In [19]:
pos_freq_dist=FreqDist()
neg_freq_dist=FreqDist()

for reviewDist,label in training_norm:
    if label=='pos':
        pos_freq_dist+=reviewDist
    else:
        neg_freq_dist+=reviewDist

neg_freq_dist

FreqDist({'film': 2917, 'movie': 2222, 'one': 1938, 'like': 1306, 'even': 978, 'good': 809, 'time': 808, 'would': 744, 'get': 719, 'much': 706, ...})

### Exercise 3.2
In the blank code cell below write code that uses the total frequency distributions `pos_freq_dist` and `neg_freq_dist` and the word lists `my_positive_word_list` and `my_negative_word_list` created earlier to determine whether or not the review data conforms to your expectations. In particular, whether:
- the words you expected to indicate positive sentiment actually occur more frequently in positive reviews than negative reviews
- the words you expected to indicate negative sentiment actually occur more frequently in negative reviews than positive reviews.

You could display your findings in a table using pandas.

In [20]:
results = []

for word in my_positive_word_list:
    if pos_freq_dist[word] > neg_freq_dist[word]:
        expect = "yes"
    else:
        expect = "no"
    results.append((word, expect))

for word in my_negative_word_list:
    if neg_freq_dist[word] > pos_freq_dist[word]:
        expect = "yes"
    else:
        expect = "no"
    results.append((word, expect))

print(results)

[('good', 'yes'), ('great', 'yes'), ('lovely', 'no'), ('excellent', 'yes'), ('amazing', 'yes'), ('wonderful', 'yes'), ('fantastic', 'yes'), ('brilliant', 'yes'), ('superb', 'yes'), ('outstanding', 'yes'), ('masterpiece', 'yes'), ('awesome', 'yes'), ('beautiful', 'yes'), ('perfect', 'yes'), ('enjoyable', 'yes'), ('funny', 'no'), ('love', 'yes'), ('hilarious', 'yes'), ('engaging', 'yes'), ('smart', 'yes'), ('bad', 'yes'), ('terrible', 'yes'), ('awful', 'yes'), ('boring', 'yes'), ('worst', 'yes'), ('waste', 'yes'), ('dreadful', 'yes'), ('horrible', 'yes'), ('stupid', 'yes'), ('dull', 'yes'), ('disappointing', 'yes'), ('pointless', 'yes'), ('annoying', 'yes'), ('mess', 'yes'), ('poor', 'yes'), ('hate', 'yes'), ('badly', 'yes'), ('lame', 'yes'), ('predictable', 'yes'), ('cliche', 'yes')]


In [21]:
import pandas as pd

tbl = pd.DataFrame(results,columns=["Word", "Conforms"])
tbl



,Word,Conforms
0,good,yes
1,great,yes
2,lovely,no
3,excellent,yes
4,amazing,yes
5,wonderful,yes
6,fantastic,yes
7,brilliant,yes
8,superb,yes
9,outstanding,yes


### Exercise 3.3
Now, you are going to create positive and negative word lists automatically from the training data. In order to do this:

1. write two new functions to help with automating the process of generating wordlists.

    - `most_frequent_words` - this function should take THREE arguments: 2 frequency distributions and a natural number, k. It should order words by how much more they occur in one frequency distribution than the other.   It should then return the top k highest scoring words. You might want to use the `most_common` method from the `FreqDist` class - this returns a list of word, frequency pairs ordered by frequency.  You might also or alternatively want to use pythons built-in `sorted` function
    - `words_above_threshold` - this function also takes three arguments: 2 frequency distributions and a natural number, k. Again, it should order words by how much more they occur in one distribution than the other.  It should return all of the words that have a score greater than k.

2. Using the training data, create two sets of positive and negative word lists using these functions (1 set with each function).
3.  Display these 4 lists (possibly in a `Pandas` dataframe?)



In [22]:
def most_frequent_words(freq1, freq2, k):
  newdist = freq1-freq2
  topk=[word for word, freq in newdist.most_common(k)]
  return topk

most_frequent_words(pos_freq_dist,neg_freq_dist,10)

['film',
 'life',
 'also',
 'great',
 'best',
 'well',
 'many',
 'story',
 'world',
 'one']

## Creating a word list based classifier
Now you have a number of word lists for use with a classifier.
> Make sure you understand the following code, which will be used as the basis for creating a word list based classifier.

In [23]:
from nltk.classify.api import ClassifierI
import random

class SimpleClassifier(ClassifierI):

    def __init__(self, pos, neg): #init is initializer
        self._pos = pos
        self._neg = neg

    def classify(self, words):
        score = 0

        # add code here that assigns an appropriate value to score
        return "neg" if score < 0 else "pos"

    ##we don't actually need to define the classify_many method as it is provided in ClassifierI
    #def classify_many(self, docs):
    #    return [self.classify(doc) for doc in docs]

    def labels(self):
        return ("pos", "neg")

#Example usage:

classifier = SimpleClassifier(my_positive_word_list, my_negative_word_list)
classifier.classify(FreqDist("This movie was great".split()))

'pos'

### Exercise 3.1

- Copy the above code cell and move it to below this one. Then complete the `classify` method in the above code as specified below.
- Test your classifier on several very simple hand-crafted examples to verify that you have implemented `classify` correctly.

The classifier is initialised with a list of positive words, and a list of negative words. The words of a document are passed to the `classify` method (which is partially completed in the above code fragment). The `classify` method should be defined so that each occurrence of a negative word decrements `score`, and each occurrence of a positive word increments `score`.
- For `score` less than 0, "`neg`" for negative should be returned.
- For `score` greater than 0,  "`pos`" for positive should returned.
- For `score` of 0, the classification decision should be made randomly (see https://docs.python.org/3/library/random.html).


In [24]:
import random
from nltk.classify.api import ClassifierI
from nltk.probability import FreqDist

class SimpleClassifier(ClassifierI): #inherits class functions from nltk and only write cod for the new features

    def __init__(self, pos, neg):
        self._pos = pos
        self._neg = neg

    def classify(self, words):
        score = 0

        # Simply count positive and negative words
        for word in words:
            if word in self._pos:
                score += 1
            elif word in self._neg:
                score -= 1

        if score > 0:
            return "pos"
        elif score < 0:
            return "neg"
        else:
            return random.choice(["pos", "neg"])

    def labels(self):
        return ("pos", "neg")

# Example usage:
classifier = SimpleClassifier(my_positive_word_list, my_negative_word_list)
print(classifier.classify(FreqDist("This movie was bad".split())))

neg


### Exercise 3.2
* Extend your SimpleClassifier class so that it has a `train` function which will derive the wordlists from training data.  You could build a separate class for each way of automatically deriving wordlists (which both inherit from SimpleClassifier) OR a single class which takes an extra parameter at training time.

In [27]:


# Standard base classifier from the previous exercises
class SimpleClassifier(ClassifierI):

    def __init__(self, pos, neg):
        self._pos = pos
        self._neg = neg

    def classify(self, words):
        score = 0
        for word in words:
            if word in self._pos:
                score += 1
            elif word in self._neg:
                score -= 1

        if score > 0:
            return "pos"
        elif score < 0:
            return "neg"
        else:
            return random.choice(["pos", "neg"])

    def labels(self):
        return ("pos", "neg")

# Trained subclass that inherits from SimpleClassifier
class WordListClassifier(SimpleClassifier):

    def __init__(self, k=100):
        # Initialize with empty lists; train() will populate them
        super().__init__([], [])
        self.k = k

    def train(self, training_data):
        # 1. Create frequency distributions for positive and negative training items
        pos_freq = FreqDist()
        neg_freq = FreqDist()

        for doc_dist, label in training_data:
            if label == 'pos':
                pos_freq += doc_dist
            else:
                neg_freq += doc_dist

        # 2. Derive the lists using the differences in word frequencies
        # Positive list: words appearing more in pos_freq than neg_freq
        pos_diff = pos_freq - neg_freq
        self._pos = [word for word, freq in pos_diff.most_common(self.k)]

        # Negative list: words appearing more in neg_freq than pos_freq
        neg_diff = neg_freq - pos_freq
        self._neg = [word for word, freq in neg_diff.most_common(self.k)]

# Example usage to train and test:
trained_classifier = WordListClassifier(k=100)
trained_classifier.train(training_norm)

print("Top 5 trained positive words:", trained_classifier._pos[:5])
print("Top 5 trained negative words:", trained_classifier._neg[:5])

Top 5 trained positive words: ['film', 'life', 'also', 'great', 'best']
Top 5 trained negative words: ['movie', 'bad', 'plot', 'even', 'worst']


Try out your classifier on the test data.  We will look at how to evaluate classifiers in the next part, but in an ideal world, most of the positive test items will have been classified as 'P' and most of the negative test items will have been classified as 'N'.  Note that the batch_classify method takes a list of unlabelled documents so you can't give it a list of pairs (where each pair is doc and a label).  You can either use a list comprehension or the <code>zip(*list_of_pairs)</code> function to split a list of pairs into a pair of lists.

In [28]:
# 1. Unzip the test pairs into two lists: test_docs (unlabelled documents) and test_labels (true tags)
test_docs, test_labels = zip(*testing_norm)

# 2. Use batch_classify (classify_many) to run predictions on all test docs
predictions = trained_classifier.classify_many(test_docs)

# 3. Analyze how many are classified correctly
pos_correct = 0
neg_correct = 0
pos_total = 0
neg_total = 0

for pred, actual in zip(predictions, test_labels):
    if actual == 'pos':
        pos_total += 1
        if pred == 'pos':
            pos_correct += 1
    elif actual == 'neg':
        neg_total += 1
        if pred == 'neg':
            neg_correct += 1

print(f"Correctly classified Positive items: {pos_correct} / {pos_total} ({pos_correct/pos_total:.1%})")
print(f"Correctly classified Negative items: {neg_correct} / {neg_total} ({neg_correct/neg_total:.1%})")

Correctly classified Positive items: 286 / 300 (95.3%)
Correctly classified Negative items: 123 / 300 (41.0%)


The code above shows bias towards the majority class (positive). The positive class dominates if there is a minor overlaps

In [29]:
# This is our new, smarter classifier that inherits from SimpleClassifier
class ImprovedWordListClassifier(SimpleClassifier):

    # Step 1: Set up the classifier with standard default options
    def __init__(self, k=100, ratio_threshold=1.5):
        # Start with empty positive/negative word lists; training will fill these in
        super().__init__([], [])
        self.k = k                      # How many top words we want to keep (default 100)
        self.ratio_threshold = ratio_threshold  # How much more frequent a word must be in one category to be chosen

    # Step 2: Extract the best sentiment words from the training reviews
    def train(self, training_data):
        # Create empty counters to count word frequencies
        pos_freq = FreqDist()
        neg_freq = FreqDist()

        # Count every word in the positive and negative training reviews
        for doc_dist, label in training_data:
            if label == 'pos':
                pos_freq += doc_dist
            else:
                neg_freq += doc_dist

        # Placeholders to hold candidate words that pass our test
        pos_candidates = {}
        neg_candidates = {}

        # Create a combined master list of all unique words across both collections
        all_words = set(pos_freq.keys()).union(set(neg_freq.keys()))

        # Go through each word to see if it is distinctly positive or negative
        for word in all_words:
            p_count = pos_freq[word] # How many times it appears in positive reviews
            n_count = neg_freq[word] # How many times it appears in negative reviews

            # If it appears more in positive reviews:
            if p_count > n_count:
                # Calculate the ratio (we add +1 to avoid dividing by zero errors)
                ratio = (p_count + 1) / (n_count + 1)
                # Only keep the word if it is significantly more common in positive reviews
                if ratio >= self.ratio_threshold:
                    pos_candidates[word] = p_count - n_count

            # If it appears more in negative reviews:
            elif n_count > p_count:
                # Calculate the ratio
                ratio = (n_count + 1) / (p_count + 1)
                # Only keep the word if it is significantly more common in negative reviews
                if ratio >= self.ratio_threshold:
                    neg_candidates[word] = n_count - p_count

        # Sort the distinctive words by their frequency difference and take the top k
        self._pos = sorted(pos_candidates, key=pos_candidates.get, reverse=True)[:self.k]
        self._neg = sorted(neg_candidates, key=neg_candidates.get, reverse=True)[:self.k]

# --- Let's run and test this improved version! ---

# 1. Train the classifier using our 1.5 ratio threshold
improved_classifier = ImprovedWordListClassifier(k=100, ratio_threshold=1.5)
improved_classifier.train(training_norm)

# Print the top 10 positive and negative words to inspect them
print("Improved Positive words:", improved_classifier._pos[:10])
print("Improved Negative words:", improved_classifier._neg[:10])

# 2. Get predictions for all the test reviews
predictions_improved = improved_classifier.classify_many(test_docs)

# 3. Calculate and print accuracy percentages
pos_correct, neg_correct = 0, 0
pos_total, neg_total = 0, 0

for pred, actual in zip(predictions_improved, test_labels):
    if actual == 'pos':
        pos_total += 1
        if pred == 'pos':
            pos_correct += 1
    elif actual == 'neg':
        neg_total += 1
        if pred == 'neg':
            neg_correct += 1

print(f"\nImproved Positive accuracy: {pos_correct} / {pos_total} ({pos_correct/pos_total:.1%})")
print(f"Improved Negative accuracy: {neg_correct} / {neg_total} ({neg_correct/neg_total:.1%})")
print(f"Overall Balanced accuracy: {(pos_correct + neg_correct) / (pos_total + neg_total):.1%}")

Improved Positive words: ['life', 'also', 'great', 'best', 'many', 'world', 'performance', 'american', 'family', 'war']
Improved Negative words: ['bad', 'plot', 'worst', 'script', 'stupid', 'supposed', 'godzilla', 'harry', 'boring', 'minutes']

Improved Positive accuracy: 293 / 300 (97.7%)
Improved Negative accuracy: 128 / 300 (42.7%)
Overall Balanced accuracy: 70.2%
